# Notebook 01 — Single-Sample SGD with MSE

**Session:** Linear Regression — L1, L2, and Huber Losses (Level 2)  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement stochastic gradient descent (SGD) for a single-feature linear model using the
Mean Squared Error (MSE) loss function.  We train on a 4-sample toy housing dataset where
the feature is floor area (×100 m²) and the target is price (Tael).

**Learning objectives:**
- Load a headerless CSV file with NumPy.
- Implement `predict`, `compute_loss_mse`, `gradient_mse`, and `update_weight` from scratch.
- Run the SGD loop and observe the per-sample loss trajectory.
- Plot the fitted line over the scatter plot of observations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 1  Load Data

In [ ]:
def get_column(data: list, index: int) -> list:
    """Return the column at *index* from a 2-D list."""
    return [row[index] for row in data]


data = np.genfromtxt("../data/data.csv", delimiter=",").tolist()

x_data = get_column(data, 0)  # area
y_data = get_column(data, 1)  # price
N = len(x_data)

print(f"areas  : {x_data}")
print(f"prices : {y_data}")
print(f"N      : {N}")

### Scatter plot of observations

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(x_data, y_data)
plt.xlabel("Area (×100 m²)")
plt.ylabel("Price (Tael)")
plt.xlim(3, 7)
plt.ylim(4, 10)
plt.title("Toy Housing Dataset")
plt.tight_layout()
plt.show()

## 2  Model and Loss Functions

In [ ]:
def predict(x: float, w: float, b: float) -> float:
    """Linear prediction: y_hat = w * x + b."""
    return w * x + b


def gradient_mse(y_hat: float, y: float, x: float) -> tuple[float, float]:
    """Per-sample MSE gradients with respect to w and b.

    dL/dw = 2 * x * (y_hat - y)
    dL/db = 2 * (y_hat - y)
    """
    dw = 2 * x * (y_hat - y)
    db = 2 * (y_hat - y)
    return dw, db


def update_weight(w: float, b: float, lr: float, dw: float, db: float) -> tuple[float, float]:
    """Gradient descent weight update."""
    return w - lr * dw, b - lr * db

## 3  SGD Training Loop

In [ ]:
# --- Hyperparameters ---
w: float = -0.34
b: float = 0.04
lr: float = 0.02
epoch_max: int = 10

losses: list[float] = []

for _epoch in range(epoch_max):
    for i in range(N):
        x = x_data[i]
        y = y_data[i]

        y_hat = predict(x, w, b)
        loss = (y_hat - y) ** 2
        losses.append(loss)

        dw, db = gradient_mse(y_hat, y, x)
        w, b = update_weight(w, b, lr, dw, db)

print(f"Final parameters  →  w = {w:.4f}, b = {b:.4f}")

## 4  Inspect Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses)
axes[0].set_title("Per-sample MSE loss (all iterations)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("MSE loss")

axes[1].plot(losses[N:])  # skip first noisy epoch
axes[1].set_title("Per-sample MSE loss (epochs 2 onward)")
axes[1].set_xlabel("Iteration")

plt.tight_layout()
plt.show()

## 5  Fitted Line

In [ ]:
x_line = list(range(2, 8))
y_line = [predict(x, w, b) for x in x_line]

plt.figure(figsize=(6, 4))
plt.scatter(x_data, y_data, label="Observations")
plt.plot(x_line, y_line, "r-", label=f"Fit: y = {w:.3f}x + {b:.3f}")
plt.xlabel("Area (×100 m²)")
plt.ylabel("Price (Tael)")
plt.title("Linear Regression — MSE (SGD)")
plt.xlim(3, 7)
plt.ylim(4, 10)
plt.legend()
plt.tight_layout()
plt.show()

## 6  Inference

In [ ]:
area_new = 6.0
predicted_price = predict(area_new, w, b)
print(f"Predicted price for area = {area_new} → {predicted_price:.4f} Tael")